# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id
print("All record sets (referenced by @id):")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id} | Name: {record_set.name}")

# For each record set, list its fields (and corresponding @id)
for record_set in metadata.record_sets:
    print(f"\nRecord Set: {record_set.name} (@id: {record_set.id})")
    if hasattr(record_set, 'fields') and record_set.fields:
        print(f"  Fields:")
        for field in record_set.fields:
            print(f"    - @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
    # Also list the columns, if present
    if hasattr(record_set, 'columns') and record_set.columns:
        print(f"  Columns:")
        for column in record_set.columns:
            print(f"    - @id: {column.id} | Name: {column.name} | Data type: {column.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids
record_sets_ids = [rs.id for rs in metadata.record_sets]
print("Record set @ids:", record_sets_ids)

# Load each record set into a DataFrame
dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")
    else:
        print(f"No records found for record set @id: {record_set_id}")

# Show columns for the first record set loaded
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame for {example_record_set_id}:\n", dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example EDA on the first available record set
import numpy as np

if dataframes:
    # Choose a record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    
    print(f"Performing EDA on record set @id: {record_set_id}")
    # Try to identify a numeric field: choose the first column with numeric dtype
    numeric_field = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field = col
            break
    if numeric_field is not None:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > mean ({threshold:.2f}):")
        display(filtered_df.head())

        # Normalize the column
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a categorical column if available
        group_field = None
        for col in df.columns:
            if df[col].dtype == 'object' and col != numeric_field:
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found in this record set for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    # Boxplot by group_field if available
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field} in {record_set_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides comprehensive records from mass spectrometry analysis of extracellular vesicles derived from stimulated human mast cells.
- Data fields include various numeric and categorical attributes suitable for downstream bioinformatics analysis.
- Using Croissant's `@id` fields ensures reliable referencing of record sets and fields.
- The exploration above demonstrates how to programmatically load, filter, normalize, and visualize data for further research.

Further analysis can be performed based on the specific biological questions or by integrating with other proteomics datasets.